<a href="https://colab.research.google.com/github/back1992/ai_math/blob/main/parse_with_gemini.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI Math Question Parser with Gemini

This notebook uses Google Gemini to automatically parse math worksheet images and extract structured questions with Chinese translations.

In [ ]:
# Install required packages
!pip install -q google-generativeai pillow

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Configure paths - UPDATE THESE TO YOUR PATHS
PROJECT_PATH = '/content/drive/MyDrive/project/ai_math'
IMAGES_FOLDER = f'{PROJECT_PATH}/output/images'
OUTPUT_JSON = f'{PROJECT_PATH}/output/part01_zh_kp.json'
PROGRESS_JSON = f'{PROJECT_PATH}/progress.json'

# Or use local paths if files are uploaded to Colab
# IMAGES_FOLDER = '/content/output/images'
# OUTPUT_JSON = '/content/output/part01_zh_kp.json'

In [ ]:
import google.generativeai as genai
from google.colab import userdata
import json
import os
from PIL import Image

# Configure Gemini API
# Get your API key from https://makersuite.google.com/app/apikey
# Store it in Colab Secrets as 'GEMINI_API_KEY'
try:
    GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
except:
    # Or paste your API key here (not recommended for sharing)
    GEMINI_API_KEY = input('Enter your Gemini API key: ')

genai.configure(api_key=GEMINI_API_KEY)
model = genai.GenerativeModel('gemini-1.5-flash')

print("✅ Gemini configured successfully")

In [ ]:
# Define the parsing prompt
PARSING_PROMPT = """
Analyze this math worksheet page carefully and extract all information.

This page is from an educational math workbook. Extract:

1. **Topic**: The main subject or invention being discussed (e.g., "Inventions: Slinky")
2. **Context**: The problem statement or background information
3. **Clues**: All numbered clues or constraints (if present)
4. **Questions**: All numbered questions asked to students
5. **Knowledge Points**: Math concepts covered (e.g., "Algebraic Reasoning", "Factors and Multiples", "Inequalities")

Return ONLY a valid JSON object with this exact structure:
{
  "topic_en": "Topic in English",
  "topic_zh": "主题的中文翻译",
  "context_en": "Context/problem statement in English",
  "context_zh": "背景/问题陈述的中文翻译",
  "clues_en": ["1) First clue", "2) Second clue"],
  "clues_zh": ["1) 第一条线索", "2) 第二条线索"],
  "questions_en": ["1. First question?", "2. Second question?"],
  "questions_zh": ["1. 第一个问题？", "2. 第二个问题？"],
  "knowledge_points": ["Concept 1", "Concept 2"]
}

IMPORTANT:
- If there are no clues, use empty arrays: "clues_en": [], "clues_zh": []
- Preserve mathematical notation exactly (e.g., >=, <=, ≠, ÷, ×)
- Include all questions, even if partially visible
- Provide accurate Chinese translations
- Return ONLY the JSON, no other text
"""

In [ ]:
def parse_page_with_gemini(image_path, page_num):
    """
    Use Gemini to parse a page image and extract structured questions
    
    Args:
        image_path: Path to the page image
        page_num: Page number
    
    Returns:
        Dictionary with structured question data
    """
    try:
        # Load image
        img = Image.open(image_path)
        
        # Generate content with Gemini
        response = model.generate_content([PARSING_PROMPT, img])
        
        # Extract JSON from response
        response_text = response.text.strip()
        
        # Remove markdown code blocks if present
        if response_text.startswith('```json'):
            response_text = response_text[7:]
        if response_text.startswith('```'):
            response_text = response_text[3:]
        if response_text.endswith('```'):
            response_text = response_text[:-3]
        
        response_text = response_text.strip()
        
        # Parse JSON
        parsed_data = json.loads(response_text)
        
        # Add page number
        parsed_data['page'] = page_num
        
        return parsed_data
        
    except json.JSONDecodeError as e:
        print(f"❌ JSON parsing error for page {page_num}: {e}")
        print(f"Response: {response_text[:200]}...")
        return None
    except Exception as e:
        print(f"❌ Error parsing page {page_num}: {e}")
        return None

In [ ]:
def load_progress():
    """Load progress tracking data"""
    if os.path.exists(PROGRESS_JSON):
        with open(PROGRESS_JSON, 'r', encoding='utf-8') as f:
            return json.load(f)
    return {
        "completed_pages": [],
        "skipped_pages": []
    }

def load_existing_questions():
    """Load existing parsed questions"""
    if os.path.exists(OUTPUT_JSON):
        with open(OUTPUT_JSON, 'r', encoding='utf-8') as f:
            return json.load(f)
    return []

def save_questions(questions):
    """Save questions to JSON file"""
    # Sort by page number
    questions.sort(key=lambda x: x['page'])
    
    with open(OUTPUT_JSON, 'w', encoding='utf-8') as f:
        json.dump(questions, f, ensure_ascii=False, indent=2)
    
    print(f"✅ Saved {len(questions)} questions to {OUTPUT_JSON}")

In [ ]:
def process_pages_batch(start_page, end_page):
    """
    Process a batch of pages with Gemini
    
    Args:
        start_page: Starting page number
        end_page: Ending page number (inclusive)
    """
    # Load existing data
    progress = load_progress()
    existing_questions = load_existing_questions()
    existing_pages = {q['page'] for q in existing_questions}
    
    print(f"\n🚀 Processing pages {start_page}-{end_page}")
    print(f"📊 Already have {len(existing_questions)} pages parsed\n")
    
    newly_parsed = []
    
    for page_num in range(start_page, end_page + 1):
        # Skip if already parsed
        if page_num in existing_pages:
            print(f"⏭️  Page {page_num}: Already parsed, skipping")
            continue
        
        # Skip if in skipped list
        if page_num in progress.get('skipped_pages', []):
            print(f"⏭️  Page {page_num}: Marked as skipped")
            continue
        
        # Check if image exists
        image_path = f"{IMAGES_FOLDER}/page_{page_num}.jpg"
        if not os.path.exists(image_path):
            print(f"⚠️  Page {page_num}: Image not found at {image_path}")
            continue
        
        print(f"📄 Processing page {page_num}...")
        
        # Parse with Gemini
        parsed = parse_page_with_gemini(image_path, page_num)
        
        if parsed:
            newly_parsed.append(parsed)
            print(f"   ✅ Successfully parsed page {page_num}")
            print(f"   📝 Topic: {parsed.get('topic_en', 'N/A')[:50]}...")
        else:
            print(f"   ❌ Failed to parse page {page_num}")
        
        print()
    
    # Merge with existing questions
    all_questions = existing_questions + newly_parsed
    
    # Remove duplicates (keep newer version)
    questions_dict = {q['page']: q for q in all_questions}
    final_questions = list(questions_dict.values())
    
    # Save
    if newly_parsed:
        save_questions(final_questions)
        print(f"\n✅ Parsed {len(newly_parsed)} new pages")
        print(f"📊 Total pages in output: {len(final_questions)}")
    else:
        print("\n⚠️  No new pages were parsed")
    
    return newly_parsed

## Test with a Single Page

Let's test with page 14 first to make sure everything works:

In [ ]:
# Test with page 14
test_page = 14
test_image = f"{IMAGES_FOLDER}/page_{test_page}.jpg"

if os.path.exists(test_image):
    print(f"Testing with page {test_page}...\n")
    result = parse_page_with_gemini(test_image, test_page)
    
    if result:
        print("✅ Success! Parsed data:")
        print(json.dumps(result, indent=2, ensure_ascii=False))
    else:
        print("❌ Failed to parse")
else:
    print(f"❌ Image not found: {test_image}")
    print(f"\nPlease update IMAGES_FOLDER path or upload images to Colab")

## Process Batch of Pages

Now process pages 14-23 (or any range you want):

In [ ]:
# Process pages 14-23
START_PAGE = 14
END_PAGE = 23

results = process_pages_batch(START_PAGE, END_PAGE)

## Process All Remaining Pages

Process all pages from 14 to 90:

In [ ]:
# Process all pages from 14 to 90
# WARNING: This will use API credits - process in batches if needed

# Process in batches of 10 to avoid rate limits
for batch_start in range(14, 91, 10):
    batch_end = min(batch_start + 9, 90)
    print(f"\n{'='*60}")
    print(f"Processing batch: pages {batch_start}-{batch_end}")
    print(f"{'='*60}")
    
    results = process_pages_batch(batch_start, batch_end)
    
    # Small delay between batches to avoid rate limits
    import time
    time.sleep(2)

## View Results

Check the parsed questions:

In [ ]:
# Load and display results
questions = load_existing_questions()

print(f"📊 Total parsed pages: {len(questions)}")
print(f"\nPages: {sorted([q['page'] for q in questions])}")

# Show sample
if questions:
    print(f"\n📄 Sample (Page {questions[0]['page']}):")
    print(json.dumps(questions[0], indent=2, ensure_ascii=False)[:500])
    print("...")

## Download Results

Download the parsed questions JSON file:

In [ ]:
from google.colab import files

# Download the output file
if os.path.exists(OUTPUT_JSON):
    files.download(OUTPUT_JSON)
    print(f"✅ Downloaded {OUTPUT_JSON}")
else:
    print(f"❌ File not found: {OUTPUT_JSON}")